In [ ]:
# ============================================================
# LSTM TRAINING SCRIPT
# - optimized for MacBook Air / Apple Silicon
# - nested CV + Optuna, but lighter than the original cluster script
# - saves ONE single PKL per region containing:
#     * fitted final Darts LSTM model
#     * target scaler
#     * covariate scaler
#     * best params
#     * fold summaries
#     * full Optuna trial table
#     * metadata needed for later reuse / export
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import json
import pickle
import warnings
import datetime
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import optuna

import lightning.pytorch as pl
sys.modules["pytorch_lightning"] = pl

from optuna.integration.pytorch_lightning import PyTorchLightningPruningCallback
from lightning.pytorch.callbacks.early_stopping import EarlyStopping

from sklearn.preprocessing import StandardScaler

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import Scaler
from darts.metrics import rmse
from darts.models import BlockRNNModel


In [ ]:


# ============================================================
# CONFIG
# ============================================================

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

# ----- PATHS -----
WORK_DIR = Path("../Papermethods/")
INPUT_DATA_LOC = Path("../EDA/")
RESULTS_DIR = Path("../Outputs/lstm/")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

INPUT_CSV = INPUT_DATA_LOC / "region_temp_extndd.csv"

# ----- COMPUTE -----
N_THREADS = 4
os.environ["OMP_NUM_THREADS"] = str(N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(N_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(N_THREADS)
os.environ["TORCH_NUM_THREADS"] = str(N_THREADS)
torch.set_num_threads(N_THREADS)
torch.set_float32_matmul_precision("medium")

# ----- DATA / MODEL CONFIG -----
DATE_COL = "Date"
REGION_COL = "name"
TARGET_VALUE_COL = "de_trend_seas"
COVARIATE_COLS = ["dayofyear", "month", "dts_doyavge", "dts_doyvar"]

MODEL_NAME = "LSTM"

CFG = {
    "random_state": 42,
    "n_folds": 2,
    "val_len_days": 365 * 3,
    "n_trials": 3,
    "trial_epochs": 8,
    "final_epochs": 25,
    "early_stop_patience": 2,
    "early_stop_min_delta": 0.001,
    "num_workers": 0,
    "batch_size": 128,
    "forecast_horizon": 366,
    "days_in_min": 2,
    "days_in_max": 3,
    "hidden_dims": [8, 16, 32],
    "n_rnn_layers_min": 1,
    "n_rnn_layers_max": 1,
    "dropout_min": 0.0,
    "dropout_max": 0.2,
    "lr_min": 1e-4,
    "lr_max": 8e-4,
    "save_checkpoints_final": False,
}


# ============================================================
# HELPERS
# ============================================================

def get_accelerator() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


ACCELERATOR = get_accelerator()


def make_trainer_kwargs(callbacks=None):
    kwargs = {
        "accelerator": ACCELERATOR,
        "devices": 1,
        "strategy": "auto",
        "enable_progress_bar": False,
        "default_root_dir": str(RESULTS_DIR),
        "logger": False,
    }
    if callbacks is not None:
        kwargs["callbacks"] = callbacks
    return kwargs


def time_series_cv(series_ts: TimeSeries,
                   cov_ts: TimeSeries,
                   n_folds: int,
                   val_size: int):
    folds = []
    total_len = len(series_ts)

    for i in range(n_folds):
        split_point = total_len - (n_folds - i) * val_size

        train = series_ts[:split_point]
        val = series_ts[split_point:split_point + val_size]
        cov_train = cov_ts[:split_point]
        cov_val = cov_ts[split_point:split_point + val_size]

        folds.append((train, val, cov_train, cov_val))

    return folds


def build_region_dataframe(feat_df: pd.DataFrame, region):
    region_df = (
        feat_df.loc[
            feat_df[REGION_COL] == region,
            [REGION_COL, "TAVG_imptd", "trend", "detrended", "four_seas",
             "de_trend_seas", "dayofyear", "month", "dts_doyavge", "dts_doyvar"]
        ]
        .asfreq(freq="D")
        .interpolate(method="linear", limit_direction="forward", axis=0)
        .copy()
    )
    return region_df


def to_timeseries(region_df: pd.DataFrame):
    series = TimeSeries.from_dataframe(
        df=region_df.reset_index(),
        time_col=DATE_COL,
        value_cols=TARGET_VALUE_COL,
        freq="D",
        fill_missing_dates=False,
    )

    covariate = TimeSeries.from_dataframe(
        df=region_df.reset_index(),
        time_col=DATE_COL,
        value_cols=COVARIATE_COLS,
        freq="D",
        fill_missing_dates=False,
    )
    return series, covariate


def optuna_objective(trial, train, val, cov_train, cov_val):
    try:
        early_stop_callback = EarlyStopping(
            monitor="val_loss",
            patience=CFG["early_stop_patience"],
            min_delta=CFG["early_stop_min_delta"],
            mode="min",
            verbose=False,
        )
        pruning_callback = PyTorchLightningPruningCallback(trial, monitor="val_loss")

        pl_trainer_kwargs = make_trainer_kwargs(
            callbacks=[early_stop_callback, pruning_callback]
        )

        days_in = trial.suggest_int("days_in", CFG["days_in_min"], CFG["days_in_max"])
        input_chunk_length = 365 * days_in
        output_chunk_length = 1
        lr = trial.suggest_float("lr", CFG["lr_min"], CFG["lr_max"], log=True)
        hidden_dim = trial.suggest_categorical("hidden_dim", CFG["hidden_dims"])
        n_rnn_layers = trial.suggest_int(
            "n_rnn_layers",
            CFG["n_rnn_layers_min"],
            CFG["n_rnn_layers_max"],
        )
        dropout = trial.suggest_float("dropout", CFG["dropout_min"], CFG["dropout_max"])
        activation = trial.suggest_categorical("activation", ["ReLU", "tanh"])

        trial.set_user_attr("input_chunk_length", input_chunk_length)
        trial.set_user_attr("output_chunk_length", output_chunk_length)

        model = BlockRNNModel(
            model="LSTM",
            input_chunk_length=input_chunk_length,
            output_chunk_length=output_chunk_length,
            batch_size=CFG["batch_size"],
            n_epochs=CFG["trial_epochs"],
            nr_epochs_val_period=1,
            random_state=CFG["random_state"],
            hidden_dim=hidden_dim,
            n_rnn_layers=n_rnn_layers,
            dropout=dropout,
            activation=activation,
            optimizer_kwargs={"lr": lr},
            model_name=f"lstm_trial_{trial.number}",
            likelihood=None,
            force_reset=True,
            save_checkpoints=False,
            pl_trainer_kwargs=pl_trainer_kwargs,
        )

        past_covs_for_val = concatenate([cov_train, cov_val])

        model.fit(
            series=train,
            past_covariates=cov_train,
            val_series=val,
            val_past_covariates=past_covs_for_val,
            verbose=False,
            dataloader_kwargs={"num_workers": CFG["num_workers"]},
        )

        forecast = model.predict(
            n=len(val),
            series=train,
            past_covariates=past_covs_for_val,
            verbose=False,
        )

        score = rmse(val, forecast)

        print(
            f"Trial {trial.number} done | "
            f"days_in={days_in} hidden={hidden_dim} layers={n_rnn_layers} "
            f"dropout={dropout:.3f} lr={lr:.6f} | RMSE={score:.6f}"
        )
        return score

    except Exception as e:
        print(f"Trial {trial.number} failed: {repr(e)}")
        traceback.print_exc()
        return float("inf")


# ============================================================
# MAIN
# ============================================================

def main():
    feat_df = pd.read_csv(INPUT_CSV, index_col=[0], parse_dates=True)

    regions = feat_df[REGION_COL].dropna().unique()

    if len(sys.argv) > 1:
        region = feat_df[REGION_COL].unique()[int(sys.argv[1]) - 1]
        run_regions = [region]
    else:
        run_regions = regions

    print(f"Using accelerator: {ACCELERATOR}")
    print(f"Torch threads: {torch.get_num_threads()}")
    print(f"Found {len(run_regions)} region(s) to run")

    for region in run_regions:
        print(f"\n=== REGION {region} ===")

        region_raw = feat_df.loc[feat_df[REGION_COL] == region].copy()
        print(f"  Raw rows: {len(region_raw)} | target NaN: {region_raw[TARGET_VALUE_COL].isna().sum()} | "
              f"target inf: {np.isinf(region_raw[TARGET_VALUE_COL]).sum() if np.issubdtype(region_raw[TARGET_VALUE_COL].dtype, np.number) else 0}")

        region_df = build_region_dataframe(feat_df, region)
        print(f"  Clean/regularized rows: {len(region_df)}")

        series, covariate = to_timeseries(region_df)

        target_scaler = Scaler(scaler=StandardScaler())
        series_scaled = target_scaler.fit_transform(series)

        covariate_scaler = Scaler(scaler=StandardScaler())
        cov_scaled = covariate_scaler.fit_transform(covariate)

        folds = time_series_cv(
            series_scaled,
            cov_scaled,
            n_folds=CFG["n_folds"],
            val_size=CFG["val_len_days"],
        )

        best_models_info = []
        full_trial_data = pd.DataFrame()

        for i, (train, val, cov_train, cov_val) in enumerate(folds, start=1):
            print(f"  Region {region} | Fold {i}/{len(folds)} | train={len(train)} | val={len(val)}")

            study = optuna.create_study(
                direction="minimize",
                pruner=optuna.pruners.MedianPruner(
                    n_startup_trials=1,
                    n_warmup_steps=2,
                    interval_steps=1,
                ),
            )

            study.optimize(
                lambda trial: optuna_objective(trial, train, val, cov_train, cov_val),
                n_trials=CFG["n_trials"],
                show_progress_bar=False,
            )

            if study.best_trial is None:
                continue

            best_models_info.append(
                {
                    "fold": i,
                    "rmse": float(study.best_value),
                    "params": study.best_params.copy(),
                    "in_len": int(study.best_trial.user_attrs["input_chunk_length"]),
                    "out_len": int(study.best_trial.user_attrs["output_chunk_length"]),
                }
            )

            trial_rows = []
            for trial in study.trials:
                trial_rows.append(
                    {
                        "Trial": trial.number,
                        "Status": trial.state.name,
                        "RMSE": trial.value if trial.value is not None else np.nan,
                        **trial.params,
                        "fold": f"fold_{i}",
                        "region": region,
                        "model": MODEL_NAME,
                    }
                )

            full_trial_data = pd.concat(
                [full_trial_data, pd.DataFrame(trial_rows)],
                axis=0,
                ignore_index=True,
            )

        if not best_models_info:
            print(f"Skipping region {region}: no successful trials")
            continue

        best_model_info = min(best_models_info, key=lambda x: x["rmse"])
        best_params = best_model_info["params"].copy()

        best_days_in = int(best_params.pop("days_in"))
        best_lr = float(best_params.pop("lr"))
        best_input_chunk_length = int(best_model_info["in_len"])
        best_output_chunk_length = int(best_model_info["out_len"])

        print(f"  Best fold: {best_model_info['fold']} | RMSE={best_model_info['rmse']:.6f}")

        final_model = BlockRNNModel(
            model="LSTM",
            input_chunk_length=best_input_chunk_length,
            output_chunk_length=best_output_chunk_length,
            batch_size=CFG["batch_size"],
            n_epochs=CFG["final_epochs"],
            nr_epochs_val_period=1,
            random_state=CFG["random_state"],
            optimizer_kwargs={"lr": best_lr},
            model_name=f"lstm_final_{region}",
            likelihood=None,
            force_reset=True,
            save_checkpoints=CFG["save_checkpoints_final"],
            pl_trainer_kwargs=make_trainer_kwargs(),
            **best_params,
        )

        print("  Fitting final model...")
        final_model.fit(
            series=series_scaled,
            past_covariates=cov_scaled,
            verbose=False,
            dataloader_kwargs={"num_workers": CFG["num_workers"]},
        )

        forecast = final_model.predict(
            n=CFG["forecast_horizon"],
            series=series_scaled,
            past_covariates=cov_scaled,
            mc_dropout=False,
            verbose=False,
        )

        forecast_scaled_df = forecast.to_dataframe().copy()
        forecast_scaled_df[REGION_COL] = region
        forecast_scaled_df["model"] = MODEL_NAME

        forecast_unscaled = target_scaler.inverse_transform(forecast)
        forecast_unscaled_df = forecast_unscaled.to_dataframe().copy()
        forecast_unscaled_df[REGION_COL] = region
        forecast_unscaled_df["model"] = MODEL_NAME
        forecast_unscaled_df["time_taken"] = str(datetime.datetime.now() - START)

        trial_csv_path = RESULTS_DIR / f"trial_data_{region}_{TODAY}_nested_lstm.csv"
        full_trial_data.to_csv(trial_csv_path, index=False)

        scaled_forecast_csv = RESULTS_DIR / f"res_{region}_{TODAY}_scaled_nested_lstm_preds.csv"
        forecast_scaled_df.to_csv(scaled_forecast_csv, index=True)

        unscaled_forecast_csv = RESULTS_DIR / f"res_{region}_{TODAY}_unscaled_nested_lstm_preds.csv"
        forecast_unscaled_df.to_csv(unscaled_forecast_csv, index=True)

        payload = {
            "model_name": MODEL_NAME,
            "region": region,
            "created_at": datetime.datetime.now().isoformat(),
            "train_start": str(region_df.index.min().date()),
            "train_end": str(region_df.index.max().date()),
            "n_rows_raw": int(len(region_raw)),
            "n_rows_clean": int(len(region_df)),
            "target_value_col": TARGET_VALUE_COL,
            "covariate_cols": COVARIATE_COLS,
            "forecast_horizon": int(CFG["forecast_horizon"]),
            "accelerator": ACCELERATOR,
            "config": CFG,
            "best_fold": int(best_model_info["fold"]),
            "best_rmse": float(best_model_info["rmse"]),
            "best_days_in": best_days_in,
            "best_input_chunk_length": best_input_chunk_length,
            "best_output_chunk_length": best_output_chunk_length,
            "best_lr": best_lr,
            "best_params": best_params,
            "all_best_models_info": best_models_info,
            "trial_table": full_trial_data,
            "target_scaler": target_scaler,
            "covariate_scaler": covariate_scaler,
            "fitted_model": final_model,
            "forecast_scaled_df": forecast_scaled_df,
            "forecast_unscaled_df": forecast_unscaled_df,
            "trial_csv_path": str(trial_csv_path),
            "scaled_forecast_csv": str(scaled_forecast_csv),
            "unscaled_forecast_csv": str(unscaled_forecast_csv),
            "time_taken": str(datetime.datetime.now() - START),
        }

        out_pkl = RESULTS_DIR / f"lstm_region_{region}_{TODAY}.pkl"
        with open(out_pkl, "wb") as f:
            pickle.dump(payload, f)

        print(f"  Saved single model payload: {out_pkl}")

    print("\nTime taken:", datetime.datetime.now() - START)


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# EXPORT ROLLING-YEAR LSTM MODELS WITHOUT RE-TUNING
# - loads tuned per-region LSTM exports from the original run
# - expects each source PKL to be a dict payload containing:
#     * best params
#     * fitted_model
#     * target_scaler / covariate_scaler (optional, not reused directly)
# - reuses tuned hyperparameters
# - refits region models on expanding yearly samples
# - saves one PKL per region per calibration year
#
# Example:
#   calibrated through 2014-12-31 -> predicts 2015
#   calibrated through 2015-12-31 -> predicts 2016
#   ...
#   calibrated through 2023-12-31 -> predicts 2024
# ============================================================

# ============================================================
# IMPORTS
# ============================================================

import os
import sys
import pickle
import warnings
import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

import lightning.pytorch as pl
sys.modules["pytorch_lightning"] = pl

from sklearn.preprocessing import StandardScaler

from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mae, rmse, mape
from darts.models import BlockRNNModel


# ============================================================
# CONFIG
# ============================================================

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

TODAY = datetime.datetime.today().strftime("%Y-%m-%d")
START = datetime.datetime.now()

INPUT_CSV = "../EDA/region_temp_extended.csv"
SOURCE_DIR = Path("../Outputs/lstm/")
EXPORT_DIR = Path("../Outputs//lstm_rolling_exports")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)



DATE_COL = "Date"
REGION_COL = "name"
TARGET_VALUE_COL = "de_trend_seas"
COVARIATE_COLS = ["dayofyear", "month", "dts_doyavge", "dts_doyvar"]

MODEL_NAME = "LSTM"

START_CALIB_YEAR = 2014
END_CALIB_YEAR = 2023  # 2023 calibration -> 2024 prediction

RANDOM_STATE = 42
FORECAST_HORIZON = 366
FINAL_EPOCHS = 25
BATCH_SIZE = 128
NUM_WORKERS = 0

N_THREADS = 4
os.environ["OMP_NUM_THREADS"] = str(N_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(N_THREADS)
os.environ["MKL_NUM_THREADS"] = str(N_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(N_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(N_THREADS)
os.environ["TORCH_NUM_THREADS"] = str(N_THREADS)
torch.set_num_threads(N_THREADS)
torch.set_float32_matmul_precision("medium")


# ============================================================
# HELPERS
# ============================================================

def get_accelerator() -> str:
    if torch.backends.mps.is_available():
        return "mps"
    return "cpu"


ACCELERATOR = get_accelerator()


def make_trainer_kwargs():
    return {
        "accelerator": ACCELERATOR,
        "devices": 1,
        "strategy": "auto",
        "enable_progress_bar": False,
        "logger": False,
        "default_root_dir": str(EXPORT_DIR),
    }


def load_data() -> pd.DataFrame:
    feat_df = pd.read_csv(INPUT_CSV, index_col=[0], parse_dates=True)
    feat_df = feat_df.sort_index().copy()
    return feat_df


def build_region_dataframe(feat_df: pd.DataFrame, region):
    region_df = (
        feat_df.loc[
            feat_df[REGION_COL] == region,
            [REGION_COL, "TAVG_imptd", "trend", "detrended", "four_seas",
             "de_trend_seas", "dayofyear", "month", "dts_doyavge", "dts_doyvar"]
        ]
        .asfreq(freq="D")
        .interpolate(method="linear", limit_direction="forward", axis=0)
        .copy()
    )
    return region_df


def to_timeseries(region_df: pd.DataFrame):
    series = TimeSeries.from_dataframe(
        df=region_df.reset_index(),
        time_col=DATE_COL,
        value_cols=TARGET_VALUE_COL,
        freq="D",
        fill_missing_dates=False,
    )

    covariate = TimeSeries.from_dataframe(
        df=region_df.reset_index(),
        time_col=DATE_COL,
        value_cols=COVARIATE_COLS,
        freq="D",
        fill_missing_dates=False,
    )
    return series, covariate


def get_calibration_frame(region_df: pd.DataFrame, calib_year: int) -> pd.DataFrame:
    calibration_end = pd.Timestamp(f"{calib_year}-12-31")
    return region_df.loc[region_df.index <= calibration_end].copy()


def get_prediction_year_frame(region_df: pd.DataFrame, prediction_year: int) -> pd.DataFrame:
    start = pd.Timestamp(f"{prediction_year}-01-01")
    end = pd.Timestamp(f"{prediction_year}-12-31")
    return region_df.loc[(region_df.index >= start) & (region_df.index <= end)].copy()


def evaluate_predictions(y_true, y_pred):
    return {
        "MAE": float(mae(y_true, y_pred)),
        "MAPE": float(mape(y_true, y_pred)),
        "RMSE": float(rmse(y_true, y_pred)),
        "Bias": float((y_pred.values(copy=False) - y_true.values(copy=False)).mean()),
    }


def extract_tuned_info(obj: dict) -> dict:
    if not isinstance(obj, dict):
        raise ValueError("Expected source PKL to be a dict payload.")

    required = ["best_params", "best_lr", "best_input_chunk_length", "best_output_chunk_length"]
    missing = [k for k in required if k not in obj]
    if missing:
        raise ValueError(f"Source PKL missing keys: {missing}")

    return {
        "best_params": obj["best_params"],
        "best_lr": obj["best_lr"],
        "best_input_chunk_length": obj["best_input_chunk_length"],
        "best_output_chunk_length": obj["best_output_chunk_length"],
        "source_created_at": obj.get("created_at"),
        "source_best_rmse": obj.get("best_rmse"),
    }


def build_lstm_model(best_input_chunk_length, best_output_chunk_length, best_lr, best_params):
    model = BlockRNNModel(
        model="LSTM",
        input_chunk_length=best_input_chunk_length,
        output_chunk_length=best_output_chunk_length,
        batch_size=BATCH_SIZE,
        n_epochs=FINAL_EPOCHS,
        nr_epochs_val_period=1,
        random_state=RANDOM_STATE,
        optimizer_kwargs={"lr": best_lr},
        model_name="lstm_rolling_export",
        likelihood=None,
        force_reset=True,
        save_checkpoints=False,
        pl_trainer_kwargs=make_trainer_kwargs(),
        **best_params,
    )
    return model


# ============================================================
# MAIN
# ============================================================

def main():
    feat_df = load_data()

    export_rows = []
    exported_models = {}

    regions = feat_df[REGION_COL].dropna().unique()

    for region in regions:
        print(f"\n=== REGION {region} ===")

        source_candidates = sorted(SOURCE_DIR.glob(f"lstm_region_{region}_*.pkl"))
        if not source_candidates:
            print(f"Skipping {region}: no tuned source file found in {SOURCE_DIR}")
            continue

        source_pkl = source_candidates[-1]

        with open(source_pkl, "rb") as f:
            loaded_obj = pickle.load(f)

        try:
            tuned = extract_tuned_info(loaded_obj)
        except Exception as e:
            print(f"Skipping {region}: could not extract tuned info -> {e}")
            continue

        best_params = tuned["best_params"]
        best_lr = tuned["best_lr"]
        best_input_chunk_length = tuned["best_input_chunk_length"]
        best_output_chunk_length = tuned["best_output_chunk_length"]

        region_df = build_region_dataframe(feat_df, region)
        if region_df.empty:
            print(f"Skipping {region}: no usable rows")
            continue

        exported_models[str(region)] = {}

        for calib_year in range(START_CALIB_YEAR, END_CALIB_YEAR + 1):
            calibration_end = pd.Timestamp(f"{calib_year}-12-31")
            prediction_year = calib_year + 1

            calib_df = get_calibration_frame(region_df, calib_year)
            pred_year_df = get_prediction_year_frame(region_df, prediction_year)

            if len(calib_df) < max(365 * 6, best_input_chunk_length + 30):
                print(
                    f"Skipping region {region}, calib {calib_year}: "
                    f"too few calibration rows ({len(calib_df)})"
                )
                continue

            if pred_year_df.empty:
                print(
                    f"Skipping region {region}, calib {calib_year}: "
                    f"no rows for prediction year {prediction_year}"
                )
                continue

            calib_series, calib_covs = to_timeseries(calib_df)
            pred_series, pred_covs = to_timeseries(pred_year_df)

            target_scaler = Scaler(scaler=StandardScaler())
            calib_series_scaled = target_scaler.fit_transform(calib_series)

            cov_scaler = Scaler(scaler=StandardScaler())
            calib_covs_scaled = cov_scaler.fit_transform(calib_covs)

            pred_covs_scaled = cov_scaler.transform(pred_covs)

            full_covs_scaled = calib_covs_scaled.append(pred_covs_scaled)

            model = build_lstm_model(
                best_input_chunk_length=best_input_chunk_length,
                best_output_chunk_length=best_output_chunk_length,
                best_lr=best_lr,
                best_params=best_params,
            )

            model.fit(
                series=calib_series_scaled,
                past_covariates=calib_covs_scaled,
                verbose=False,
                dataloader_kwargs={"num_workers": NUM_WORKERS},
            )

            fit_scaled = model.predict(
                n=len(calib_series_scaled),
                series=calib_series_scaled,
                past_covariates=calib_covs_scaled,
                verbose=False,
            )
            fit_unscaled = target_scaler.inverse_transform(fit_scaled)

            pred_scaled = model.predict(
                n=len(pred_series),
                series=calib_series_scaled,
                past_covariates=full_covs_scaled,
                verbose=False,
            )
            pred_unscaled = target_scaler.inverse_transform(pred_scaled)

            fit_metrics = evaluate_predictions(calib_series[-len(fit_unscaled):], fit_unscaled)
            pred_year_metrics = evaluate_predictions(pred_series, pred_unscaled)

            fit_df = fit_unscaled.to_dataframe().copy()
            fit_df[REGION_COL] = region
            fit_df["model"] = MODEL_NAME

            pred_df = pred_unscaled.to_dataframe().copy()
            pred_df[REGION_COL] = region
            pred_df["model"] = MODEL_NAME

            payload = {
                "model_name": MODEL_NAME,
                "region": region,
                "calibration_year": int(calib_year),
                "calibration_end": str(calibration_end.date()),
                "prediction_year": int(prediction_year),
                "train_start": str(calib_df.index.min().date()),
                "train_end": str(calib_df.index.max().date()),
                "prediction_start": str(pred_year_df.index.min().date()),
                "prediction_end": str(pred_year_df.index.max().date()),
                "n_calibration_rows": int(len(calib_df)),
                "n_prediction_rows": int(len(pred_year_df)),
                "target_value_col": TARGET_VALUE_COL,
                "covariate_cols": COVARIATE_COLS,
                "best_params": best_params,
                "best_lr": best_lr,
                "best_input_chunk_length": best_input_chunk_length,
                "best_output_chunk_length": best_output_chunk_length,
                "fit_metrics": fit_metrics,
                "prediction_year_metrics": pred_year_metrics,
                "source_tuned_file": str(source_pkl),
                "source_best_rmse": tuned.get("source_best_rmse"),
                "target_scaler": target_scaler,
                "covariate_scaler": cov_scaler,
                "fitted_model": model,
                "fit_predictions_df": fit_df,
                "prediction_year_df": pred_df,
                "accelerator": ACCELERATOR,
                "created_at": datetime.datetime.now().isoformat(),
            }

            out_path = (
                EXPORT_DIR
                / f"lstm_region_{region}_calib_{calib_year}_predict_{prediction_year}.pkl"
            )
            with open(out_path, "wb") as f:
                pickle.dump(payload, f)

            exported_models[str(region)][int(calib_year)] = payload

            export_rows.append(
                {
                    "region": region,
                    "calibration_year": int(calib_year),
                    "calibration_end": str(calibration_end.date()),
                    "prediction_year": int(prediction_year),
                    "train_start": payload["train_start"],
                    "train_end": payload["train_end"],
                    "prediction_start": payload["prediction_start"],
                    "prediction_end": payload["prediction_end"],
                    "n_calibration_rows": payload["n_calibration_rows"],
                    "n_prediction_rows": payload["n_prediction_rows"],
                    "fit_MAE": fit_metrics["MAE"],
                    "fit_MAPE": fit_metrics["MAPE"],
                    "fit_Bias": fit_metrics["Bias"],
                    "fit_RMSE": fit_metrics["RMSE"],
                    "pred_year_MAE": pred_year_metrics["MAE"],
                    "pred_year_MAPE": pred_year_metrics["MAPE"],
                    "pred_year_Bias": pred_year_metrics["Bias"],
                    "pred_year_RMSE": pred_year_metrics["RMSE"],
                    "export_path": str(out_path),
                }
            )

            print(f"Saved region {region} | calib {calib_year} -> predict {prediction_year}")

    if export_rows:
        export_df = pd.DataFrame(export_rows)
        export_df.to_csv(
            EXPORT_DIR / f"lstm_rolling_exports_{TODAY}.csv",
            index=False,
        )

    with open(EXPORT_DIR / "all_regions_lstm_rolling_models.pkl", "wb") as f:
        pickle.dump(exported_models, f)

    print(f"\nSaved rolling calibrated model files to {EXPORT_DIR}")
    print("Time taken:", datetime.datetime.now() - START)


if __name__ == "__main__":
    main()